## **Bibliotecas**

In [1]:
#%pip install pydantic[email]

from enum import auto, IntFlag
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)

## **Classes**

In [2]:
class Role(IntFlag):
    """
    Representa diferentes posições disponíveis pros usuários.
    """

    # auto() atribui potências de 2 (1, 2, 4...) para permitir combinações únicas.
    Author = auto()     # Vale 1
    Editor = auto()     # Vale 2
    Developer = auto()  # Vale 4
    
    Admin = Author | Editor | Developer

In [3]:
class User(BaseModel):
    """
    Representa um usuário do sistema e seus dados.
    """

    # Field: Define metadados para o campo.
    name: str = Field(examples=["Arjan"])

    # EmailStr: Tipo especial que valida se é um e-mail real.
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,   # Impede que o e-mail seja alterado após a criação do objeto
    )

    # SecretStr: Protege dados sensíveis (senhas, chaves de API).
    # Se der um print(user), a senha aparecerá mascarada como '**********'.
    password: SecretStr = Field(
        examples=["Password123"],
        description="The password of the user"
    )

    # Role: Classe de posições criada anteriormente.
    role: Role = Field(
        default=None,
        description="The role of the user"
    )

## **Funções**

In [4]:
def validate(data: dict[str, Any]) -> None:
    """
    Tenta transformar um dicionário em um objeto da classe User.
    Caso os dados não sigam as regras (tipo errado, e-mail inválido, campo faltando),
    captura a exceção e detalha o erro.
    """

    try:
        # model_validate: Método do Pydantic que força a validação dos dados.
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)

## **Dados de Teste**

In [5]:
good_data = {
    "name": "Arjan",
    "email": "example@arjancodes.com",
    "password": "Password123",
}

bad_data = {
    "email": "<bad data>", 
    "password": "<bad data>"
}

## **Main**

In [6]:
validate(good_data)

name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None


In [7]:
validate(bad_data)

User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '<bad data>', 'password': '<bad data>'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '<bad data>', 'ctx': {'reason': 'An email address must have an @-sign.'}}
